# 🎧 ASR 챕터 종합 정리 노트 (2주차)

> **생성형 AI 기반 음성 에이전트 개발 과정 · ASR 파트 리뷰**
> `ASR/` 폴더의 실습 노트북 12개를 하나로 통합한 **복습·재사용용 노트**입니다.

| 항목 | 내용 |
|---|---|
| 대상 노트북 | `ASR/` 12개 (2-1 ASR원리 ~ 2-W Whisper 파인튜닝) |
| 노트의 목적 | ① 실험에 필요한 **선행 지식** ② **함수/클래스** 정의·주석 ③ **실험 진행 방법** ④ **효율적 설계 아키텍처** |
| 실행 환경 | Google Colab(T4) 실습 기준 / **macOS(Apple Silicon) 실행 가이드 포함** (2장·3장) |
| 과정 공통 표준 | 표준 발화 `utt_001~005` · 가혹조건 **SNR 5dB** · 시드 **42** · 지연 예산(왕복 **1.5s**, ASR 몫 **0.5s**) · **데이터 계약** |

> ⚙️ **실행 안내** — 이 노트의 코드 셀은 **GPU·모델 없이 실행되는** 공용 유틸·정책·어댑터만 모았습니다.
> 모델 로드가 필요한 함수는 시그니처+설명으로 요약했습니다 (2장에서 ✅/📄로 구분 표기).
> 아래에서 **위에서 아래로 순서대로** 실행하면 됩니다 (`Shift+Enter`).


## 📑 목차
| 장 | 내용 |
|---|---|
| **0** | 노트북 로드맵 (중요도·의존 관계·macOS 지원) |
| **1** | 실험에 필요한 선행 지식 (11개 주제) |
| **2** | 함수/클래스 정의 및 주석 (실행 코드 ✅ + 요약 📄) |
| **3** | 실험 진행 방법 (노트북별 가이드 + macOS 실습법) |
| **4** | 효율적 설계를 위한 아키텍처 |


# 0. 노트북 로드맵 🗺️

## 0-1. 12개 노트북 한눈에 (중요도 기준)

| 노트북 | 주제 | 중요도 | macOS 실행 | 코멘트 |
|---|---|---|---|---|
| **2-1** | ASR 원리 (멜스펙트로그램·디코딩·CER·개선기법) | ★★★ 코어 | ✅ | 모든 개념의 출발점 |
| **2-2** | 최신 ASR 비교 (faster-whisper·양자화·어댑터) | ★★★ 코어 | ✅ CPU int8 | 실무 표준 도구 |
| **2-3** | 프로덕션 ASR 모듈 (`KoreanASR`) | ★★★ 코어 | ⚠️ 모델 의존 | 2주차 최종 산출물 |
| 2-F | 파형→멜스펙트로그램 밑바닥 구현 | ★★ 기초 | ✅ GPU 불필요 | DSP 원리 (6개 엔진 공통 입력) |
| 2-R | 스트리밍 ASR (VAD·LocalAgreement) | ★★ 파이프라인 | ✅ 시뮬레이션 | 실시간 설계·6주차 연결 |
| 2-D | 화자 분리·다이어라이제이션 | ★★ 파이프라인 | ✅ CPU | DER 지표·화자별 스크립트 |
| 2-V | SenseVoice 리치 전사 (감정·이벤트) | ★★ 엔진 | ✅ GGUF 지원 | NAR 속도·확장 필드 |
| 2-W | Whisper turbo 한국어 파인튜닝 | ★★ 엔진 | ✅ CT2 int8 | 한국어 전용 체크포인트 |
| 2-S | Qwen3-ASR (LALM·LID) | ★ 엔진 | ⚠️ 비공식 | 코드 스위칭 강점 |
| 2-C | Cohere Transcribe | △ 축소(지식만) | ✖ 부적합 | 모델 카드 '한계 검증' 방법론만 |
| 2-L | GPT-Live full-duplex | ★★ 아키텍처 | ✅ 시뮬레이션 | 지연·턴테이킹 설계 |
| 2-M | Meta Omnilingual | ✖ 제외 | ✖ T4·fairseq2 전용 | 참고 1줄 (2-3 함수 목록) |

> **축소/제외 기준**: macOS 실행이 어렵거나(2-C 게이티드 2B, 2-M fairseq2/T4 전용), 한국어·실전 가치가 낮은 것은 상세에서 제외했습니다. 아키텍처·평가 지식은 유지.

## 0-2. 학습 흐름 (의존 관계)

```
2-F(음향·멜스펙트로그램 밑바닥)
   │
2-1(ASR 원리: 입력→구조→디코딩→CER→개선)
   │
   ├── 2-2(엔진 비교: faster-whisper·양자화·어댑터) ──→ 2-S/2-C/2-M(신엔진 체험)
   │            └───────────────────────────────→ 2-W(파인튜닝·양자화 심화)
   │
2-3(프로덕션 모듈: KoreanASR) ──→ 2-R(스트리밍 확장) ──→ [6주차 LiveKit/Pipecat]
   │
2-D(화자 다이어라이제이션) · 2-V(리치 전사) = 실전 부가 기능
2-L(full-duplex = 3세대 아키텍처 관점) = 왜 캐스케이드를 배우는가
```

## 0-3. 과정 공통 '표준' 4종 (모든 실험의 무대)

1. **표준 발화 `utt_001~005`** — 콜센터 도메인 5문장 (2-3 `UTTERANCES`). 같은 문장으로 모든 엔진을 채점해야 비교가 성립.
2. **가혹조건 SNR 5dB** — 콜센터 소음 환경 근사. 깨끗한 발화만으로는 실전 성능을 모름.
3. **시드 42** — gTTS 합성·노이즈 혼합의 재현성 (환경마다 결과는 달라도 **절차**는 동일).
4. **지연 예산** — 왕복 1.5s 중 ASR 몫 **0.5s(500ms)**. 예산을 넘는 정확도는 사용자에게 도달하지 못함.


## 📖 0-A. 용어 사전 & 배경 지식 — 이 노트를 처음 읽는 사람을 위한 지도

> **이 노트를 처음 공부하는 방법**: ① 0-A 용어사전을 한 번 훑어 "말이 되는지" 확인 → ② 1장 선행 지식으로
> 개념의 *이유*를 읽는다 → ③ 2장 함수를 위에서 아래로 실행하며 "검증 통과 ✅"를 눈으로 확인한다.
> 모르는 단어가 나오면 아래 사전으로 돌아오세요.

### A. 음향 신호 기초 — "소리가 숫자가 되는 과정"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 파형 (waveform) | 초당 16,000개(16kHz)의 진폭 숫자 배열 | 모든 오디오의 원형. 10초면 16만 개 숫자 |
| 준정상성 (quasi-stationarity) | 음성은 20~30ms 동안 거의 일정하다는 가정 | → 짧은 창으로 잘라야 특징이 살아남는다 |
| 프레임 / 홉 | 25ms 창 / 10ms 이동 (Whisper 표준) | 음성 → 특징 행렬로 바꾸는 첫 단계 |
| DFT / FFT | 주파수 분해. FFT는 O(N log N)으로 빠르게 | `np.fft.rfft`: 실수 신호는 절반만 계산(켤레 대칭) |
| 멜 스케일 | 사람 귀는 저주파에 민감 → 비선형 주파수 축 | 2595·log₁₀(1+f/700) |
| 멜 필터뱅크 | (n_mels, n_fft/2+1) 삼각형 행렬 | 멜 변환 = **행렬 곱 1번**. Whisper 입력 = 80채널 로그-멜 |
| 스펙트로그램 | 시간축 × 주파수축의 강도 그림 | ASR이 실제로 보는 "입력 그림" |

### B. ASR 구조 — "듣고 → 쓰는 두 개의 뇌"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 인코더 | 멜 그림을 의미 표현으로 압축하는 뇌 | 입력 전체를 한 번에 보는 부분 |
| 디코더 | 토큰을 **한 개씩** 순서대로 생성하는 뇌 | 자기회귀: 앞 글자가 다음 글자 결정 |
| 어텐션 | 지금 쓸 글자와 관련된 소리 구간에 집중 | 긴 문장에서 어디를 봐야 할지 학습 |
| 토큰 (BPE) | 글자의 최소 조각 | 한국어는 글자당 토큰이 많아 지연↑ |
| CTC / Seq2Seq | 라벨 없이 정렬 학습(CTC) vs 순서 생성(S2S) | 한국어 ASR의 두 계열 |

### C. 디코딩·신뢰도·재시도
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| greedy 디코딩 | 매번 가장 확률 높은 토큰 하나만 | 빠르지만 한 번 틀리면 되돌릴 수 없음 |
| beam search | 확률 높은 상위 k개 경로를 동시에 진행 | 정확도↑ 비용↑ — 신뢰도 낮을 때만 |
| avg_logprob | 토큰 평균 로그 확률(음수, 0에 가까울수록 확신) | 신뢰도의 양적 지표. API 엔진은 안 줌 |
| confidence_ok | 신뢰도 기준(예: logprob ≥ -0.8) 통과 여부 | 이 값이 재시도·reprompt의 스위치 |
| 재시도 정책 | 신뢰도 미달 시 beam=5로 한 번 더 | "빠른 기본 + 확실한 비상" 2단 전략 |

### D. 평가 지표 — "얼마나 잘 들었나"를 숫자로
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| CER | 글자 단위 오류율 (S+D+I)/N | 한국어는 띄어쓰기 유동 → WER 대신 CER |
| WER | 단어 단위 오류율 | 언어·도메인마다 기준이 다름 |
| RTF (Real-Time Factor) | 처리시간 / 오디오 길이. 1 미만 = 실시간 | 스트리밍 가능 여부의 분수령 |
| 지연 (latency) | 입력 후 결과까지의 시간 | 실시간 대화에서 가장 중요한 예산 |
| DER | 화자 구분 오류율 (miss+FA+confusion)/ref | 2.6에서 직접 계산 |

### E. 파이프라인·방어 장치
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| VAD (Voice Activity Detection) | "말하고 있는가"를 프레임 단위로 판정 | 무음 제거 → 비용·지연 절감 |
| EoU (End of Utterance) | 침묵 누적으로 "말이 끝났다" 판정 | 발화 분리. 0.7~1.0초 관행 |
| 환각 (hallucination) | 무음·잡음에서 모델이 지어내는 문장 | "시청해주셔서 감사합니다"류 |
| 후처리 (postprocess) | 전사 후 도메인 교정·숫자 붙이기 | 환불→환뿔 오인식 교정 |
| full-duplex | 양방향 동시 대화 (전화처럼) | barge-in(끼어들기) 수용 |

### F. 배경 지식 — 이 챕터가 왜 존재하는가
이 과정(음성 에이전트 개발)에서 **ASR은 "듣는 귀"**입니다. 뒤이어 나오는 LLM(사고), TTS(말하기),
시스템 통합(전체 조립)이 모두 **ASR이 만든 텍스트**를 입력으로 받습니다. 따라서:
1. **계약(contract)** — 어떤 ASR 엔진이든 같은 형식(`CONTRACT_KEYS`)으로 출력하게 만들어, 뒤의 LLM이
   "엔진이 바뀌어도" 같은 코드를 쓰게 합니다. 이 "계약 사상"은 이 과정 전체의 근간입니다.
2. **신뢰도 게이트** — ASR이 틀렸으면 재시도/재발화로 막고, 틀린 텍스트가 LLM·TTS까지 흘러가지 않게 합니다.
3. **가혹조건 실험** — 노이즈(SNR)·잡음·고령 화자 등 실제 환경을 미리 재현해 "언제 무너지는지"를 압니다.


### 0-B. 2장 함수 지도 — 어떤 셀이 무슨 역할인지 미리 보기
| 셀 | 함수/클래스 | 역할 한 줄 | 이 노트에서의 위치 |
|---|---|---|---|
| 2.0 | `normalize_ko`·`cer`·`add_noise_snr` | 평가·노이즈 공용 유틸 | CER/SNR 실험의 바탕 |
| 2.1 | `validate_contract` | ASR 출력의 **계약** (필수 키 검사) | 모든 엔진 공통 |
| 2.2 | `judge_confidence`·`should_retry` | 신뢰도 판정·재시도 정책 | 2단 전략 |
| 2.3 | `is_hallucination`·`postprocess_ko` | 환각 차단·한국어 교정 | 방어 장치 |
| 2.4 | `split_utterances`·`speech_ratio` | VAD 발화 분리 | 발화 단위 처리 |
| 2.5 | `stream_chunks`·`energy_vad`·`local_agreement_commit` | 스트리밍·확정 규칙 | 실시간 |
| 2.6 | `der` | 화자 구분 정확도 계산 | DER 채점 |
| 2.7 | `FullDuplexPolicy` | 전화식 양방향 의사결정 FSM | full-duplex |
| 2.8 | `BaseASR`·`AdapterFactory` | 엔진 → 표준 계약 어댑터 | 이식성 |
| 2.9 | `BenchHarness` | 여러 엔진 동일 잣대 평가 | 공정한 벤치 |
| 2.10 | `ASRConfig` | 정책 숫자를 코드에서 분리 | 설정 관리 |
| 2.11 | macOS 어댑터 | Colab과의 차이를 한 줄로 | 로컬 실행 |


# 1. 실험에 필요한 선행 지식 🧠

> 각 소주제는 **어느 노트북에서 배우는지**를 표기했습니다. 흐린 개념이 있으면 해당 노트북부터 복습하세요.

## 1-1. 음향 신호 기초 (2-F · 2-1)

| 개념 | 뜻 | 실험과의 연결 |
|---|---|---|
| **파형(waveform)** | 초당 16,000개(=16kHz)의 진폭 숫자. 10초 = 16만 개 | 파형을 그대로 모델에 주면 너무 길고 정보가 묻힘 |
| **준정상성** | 음성은 20~30ms 구간에선 거의 일정하다는 가정 | → 25ms 창으로 자르는 '프레이밍'의 근거 |
| **프레임/홉** | Whisper 표준 = 25ms 창, 10ms 이동 (n_fft=400, hop=160) | 1-3의 VAD 30ms 프레임과 같은 발상 |
| **스펙트럼 누설** | 신호를 뚝 자르면 없던 주파수가 번짐 | → 해닝 창(주기형, 분모 N)으로 완화 |
| **DFT/FFT** | 나선과의 공명으로 주파수 분해. O(N²) vs O(N log N) | `np.fft.rfft` = 실수 신호는 절반만 (켤레 대칭) |
| **멜 스케일(HTK)** | mel = 2595·log₁₀(1+f/700). 저주파는 민감, 고주파는 둔감 | 사람 귀의 감각에 맞춘 주파수 축 |
| **멜 필터뱅크** | (n_mels, n_fft/2+1) 삼각형 행렬 — 멜 변환은 **행렬 곱 1번** | Whisper 입력 = **80채널 로그-멜 스펙트로그램** |

**파이프라인 지도 (2-F)**: 파형 → ①프레이밍 → ②윈도잉 → ③DFT/FFT → ④파워 스펙트로그램(|X|²) → ⑤멜 필터뱅크 → ⑥로그 압축 = 로그 멜 스펙트로그램.

## 1-2. ASR 구조: 인코더·디코더·어텐션 (2-1)

- **인코더** = 듣는 뇌. 멜 그림(80×T)을 '의미 표현'으로 압축.
- **디코더** = 쓰는 뇌. 인코더 표현을 보며 토큰을 **한 개씩** 생성 (자기회귀).
- **어텐션** = 지금 쓸 글자와 관련된 소리 구간에 집중.
- **토큰(BPE)** = 글자의 최소 조각. 영어 데이터 편중 → 영어는 토큰 효율 좋음, 한국어는 글자당 토큰↑ → 지연↑.
- **발전사**: HMM-GMM(~2010) → DNN 하이브리드 → End-to-End(2015~) → Whisper 류(2022~, 68만시간 인터넷 학습).

> 💡 **한국어가 약한 이유 (2-W)**: Whisper 학습 데이터의 약 83%가 영어 → 한국어는 상대적으로 적은 데이터 → 파인튜닝으로 보강.


## 1-3. 디코딩 전략: greedy vs beam, 신뢰도 (2-1)

| 전략 | 방법 | 비유 | 장단점 |
|---|---|---|---|
| **Greedy** | 매 스텝 확률 1등만 | 매 갈림길에서 제일 넓어 보이는 길로 | 빠름 / 초반 실수가 끝까지 전파 |
| **Beam(k)** | 후보 k개를 나란히 유지하며 전진 | 후보 경로 k개 동시 답사 | 더 정확 / k배 느림 |

> 눈앞의 1등(A 0.6)이 문장 전체 확률을 보장하지 않는다 — A 뒤 최선 0.3 vs B 뒤 0.9. 그래서 2-3은 **평소 greedy, 신뢰도 미달 시에만 beam=5 재시도**하는 '2단 전략'을 씁니다.

**신뢰도 신호 2종** (재질문 분기·VAD 오탐 필터의 근거):
- `avg_logprob` — 토큰 평균 로그확률. 0에 가까울수록 자신감↑. 보통 **-1.0 미만이면 의심** (2-3은 -0.8 사용).
- `no_speech_prob` — '말이 없다'는 확률. **>0.6이면 무음 판정** → VAD 게이트 역할.

> ⚠️ 신뢰도는 '자신 있게 틀리는 경우'를 못 잡음(2-W) — 신뢰도 지표를 맹신하지 말 것.

## 1-4. 평가 지표: CER·WER·RTF·지연 (2-1 · 2-2)

- **CER** = (S+D+I)/N — 글자 단위 오류율. **한국어는 띄어쓰기가 유동적이라 WER 대신 CER이 표준**. 비교 전 공백·문장부호 제거(정규화) 필수.
- **편집 거리(Levenshtein)** = CER의 심장. 동적 계획법으로 최소 편집 횟수(S/D/I) 계산.
- **RTF** = 처리시간 ÷ 오디오길이. **<1.0이어야 실시간 후보**. (리더보드의 RTFx는 역수 — 단위 주의)
- **지연(latency) / 로드시간** — 1-3 지연 예산과 직결, 서비스 재시작 비용.

## 1-5. 공정한 벤치마크의 함정 (2-2)

| 함정 | 교훈 |
|---|---|
| 영어 벤치마크 1위 | 영어 1등 ≠ 한국어 1등 — **한국어는 직접 재라** |
| 깨끗한 낭독체 위주 | 낭독 성적이 시끄러운 콜센터를 대변하지 않음 → **SNR 5dB로 재라** |
| WER 기준 | 한국어는 CER로 다시 재야 함 |

> **남의 순위표는 후보 선별용, 최종 판정은 내 데이터·내 지표로.** — 벤치 하네스(2.9)가 그 도구.

## 1-6. 파이프라인 지식: VAD·EoU·환각·후처리 (1-3 · 2-3 · 2-V)

| 용어 | 뜻 | 어디서 |
|---|---|---|
| **VAD** | 말/침묵 판정 (프레임 에너지 or 신경망 Silero) | 1-3, 2-3, 2-R |
| **EoU** | 발화 종료 판정 — 침묵이 min_silence 이상 누적 | 1-3 |
| **환각** | 무음·잡음에서 그럴듯한 문장을 지어냄 (유튜브 자막 흔적) | 2-C 실험, 2-3 필터 |
| **교정 사전(lexicon)** | 반복 오인식을 결정적으로 치환 ("환뿔"→"환불") | 2-3 |
| **ITN** | 구어→문어 표기 정규화 ("삼만 원"→"30,000원") | 2-V |
| **initial_prompt vs 사전** | 전사 전 확률적 힌트 vs 전사 후 결정적 치환 — **둘 다 사용** | 2-1 vs 2-3 |

> **리치 전사(2-V)**: 한 번의 순전파로 ASR+LID+SER(감정)+AED(이벤트)를 동시에. 콜센터에서 '화난 고객 감지→에스컬레이션' 트리거 가능. NAR(비자기회귀)이라 초고속.


## 1-7. 스트리밍 ASR (2-R)

- **왜 배치로 부족한가**: RTF=0.3이라도 배치는 발화가 끝난 뒤에야 처리 → 5초 발화면 최종까지 ~6.5초 (예산의 4배).
- **핵심 구조**: 발화 *중에* 처리를 겹쳐 놓기 — "더 빠른 모델"이 아니라 **구조**가 본질.
- **VAD 엔드포인팅**: 침묵이 `silence_chunks`(3청크 × 320ms ≈ 960ms) 누적되면 발화 종료 판정.
- **LocalAgreement-2**: 연속 두 가설이 일치하는 접두어만 확정 → **확정 텍스트는 절대 뒤집히지 않음** (6주차에서 확정분은 즉시 LLM→TTS로 가므로).
- **스트리밍 3대 지표**: `first_partial_ms`(첫 부분 가설) · `final_latency_ms`(발화 종료→최종, **≤500ms**) · `rtf`(<1.0).

## 1-8. 화자 다이어라이제이션 (2-D)

| 구분 | 뜻 | 혼동 주의 |
|---|---|---|
| **다이어라이제이션** | "누가 언제 말했나" 구간 분리 | 이번 세션 주제 |
| **화자 분리(source sep.)** | 겹친 음성을 화자별 오디오로 분리 | 다른 문제 (겹침 처리) |
| **화자 인식(SPK ID)** | 목소리로 "누구인지" 식별 | 등록 DB 필요 |

- **파이프라인**: ①VAD → ②슬라이딩 윈도우(1s, 0.5s 홉) ECAPA 임베딩(192차원) → ③응집 클러스터링(코사인, 2개) → ④다수결 라벨+스무딩(300ms 미만 런 흡수) → 세그먼트.
- **DER** = (miss + false_alarm + confusion) / 참조 발화 시간.
  - miss=VAD 보수적 / FA=VAD 민감·잡음 / confusion=임베딩·군집 실패.
  - 클러스터 번호는 임의 → **최적 화자 매핑**을 찾아 채점.
- **채점 교훈**: 미니 실습 — 스무딩 끄면? WIN=500ms로 줄이면? 화자 비슷(n_steps=-2)하면 confusion↑?

## 1-9. 아키텍처 진화: 캐스케이드 → S2S → full-duplex (2-L)

```
① 캐스케이드 (본 과정 기본)   🎤 → [VAD/EoU] → [ASR] → [LLM] → [TTS] → 🔊    지연 예산 1.5s와 싸움
② 턴 기반 S2S (GPT-Realtime)  🎤 → [단일 음성 모델] → 🔊                       "네 차례 → 내 차례"
③ Full-duplex (GPT-Live)      🎤 ⇄ [연속 처리 모델] ⇄ 🔊                       차례 개념이 흐려짐
```

- **체감 공백(perceived gap)**: 턴 기반 = EoU 침묵 대기 + 파이프라인 지연 / full-duplex = 반응 프레임 수 × 40ms.
- **프레임(40ms) 정책 FSM**: R1 사용자·에이전트 동시 말함→`YIELD` / R2 2초마다→`BACKCHANNEL` / R3 침묵≥240ms→`SPEAK` / 그 외 `LISTEN`.
- **위임(delegation)**: 무거운 작업(검색·추론)은 후면 모델(GPT-5.5)로, 전면은 **filler 발화**로 공백 커버.
- **왜 캐스케이드를 배우나**: AICC 산업 현장은 관측 가능성·커스터마이즈·규제 감사 때문에 캐스케이드가 주력. full-duplex의 개념(프레임 판단·backchannel·선발화 filler)은 6주차에 캐스케이드로 이식.

## 1-10. 엔진 지형도 (2-2 · 2-S · 2-V · 2-W · 2-C)

| 엔진 | 계열 | 한국어 | macOS | 특징 |
|---|---|---|---|---|
| Whisper large-v3-turbo | 전용 ASR (ED) | ✅ 강함 | ✅ CT2/MLX | 디코더 32층→**4층**, ~6배 빠름 |
| faster-whisper | Whisper의 고속 구현 | ✅ | ✅ CPU int8 | CTranslate2·양자화, 사실상 표준 |
| 한국어 파인튜닝 turbo | faster-whisper | ✅✅ | ✅ | `ghost613/...-korean` CT2 로드 |
| Qwen3-ASR | **LALM**(LLM 기반) | 🔶 | ⚠️ | 52개 언어 LID+ASR, 코드 스위칭 강 |
| SenseVoice-Small | 전용(NAR) | 🔶 | ✅ GGUF | ASR+LID+SER+AED 리치 전사, 초고속 |
| Cohere Transcribe | Conformer ED | 🔶 | ✖ | 게이티드, 리더보드 1위(영어) — 한계 검증용 |
| Meta Omnilingual | W2V/CTC/LLM | 🔶 | ✖ | 1,672언어, fairseq2·T4 전용 |

**세 부류의 구조 구분**: ①전용 ASR(인코더 중심, 빠름) vs ②LLM 기반(이해력, 코드스위칭 강) vs ③NAR(초고속). — "이해력 vs 효율"의 2026 설계 노선.

## 1-11. macOS(Apple Silicon) 실행 가이드 🍎 (추가 조사)

> Colab(T4) 코드는 **설정 한 줄**만 바꾸면 Mac에서 재사용할 수 있습니다. T4는 CUDA GPU, Mac은 CPU/MLX(Metal)라는 차이가 전부입니다.

| 런타임 | 설치 | Mac에서의 실행 | 적합 모델 | 비고 |
|---|---|---|---|---|
| **faster-whisper** | `pip install faster-whisper` | `WhisperModel("...", device="cpu", compute_type="int8")` | base ~ large-v3-turbo, **한국어 파인튜닝 체크포인트 포함** | 기존 실습 코드와 **동일 API** → 이식 최소. int8이 fp32 대비 ~1.4배 빠름 |
| **mlx-whisper** (Apple MLX) | `pip install mlx-whisper` | `mlx_whisper.transcribe(path, path_or_hf_repo="...")` | large-v3-turbo (int4/int8 양자화) | Apple 공식 MLX, **통합 메모리**로 CPU/GPU가 데이터 이동 없이 공유. M시리즈 전용 최적화 |
| **whisper.cpp** | Homebrew/C로 빌드 | `./main -m ggml-large-v3-turbo.bin -l ko -f call.wav` | large-v3-turbo, 한국어 커뮤니티 ggml 모델 | Metal + Core ML(ANE 3x+) — **최고 속도·가장 성숙**. q5/q8 양자화. CLI 위주 |
| **SenseVoice GGUF** (llama.cpp) | GGUF 바이너리 | `sensevoice` GGUF 실행 | SenseVoice-Small (q8 ≈ 254MB) | 2-V에서 확인 — CPU/엣지·온디바이스 1차 전사. 리치 전사 유지 |

**Colab ↔ Mac 설정 치환표**

| 항목 | Colab (T4) | macOS (Apple Silicon) |
|---|---|---|
| device | `"cuda"` | `"cpu"` (MLX는 지정 불필요) |
| compute_type | `"float16"` | `"int8"` |
| GPU 메모리 해제 | `del 모델 + free_gpu()` | `del 모델` (Python GC가 처리) |
| VRAM 제약 | 16GB T4 | 통합 메모리 (16/32/64GB) |
| 모델 추천 | faster-whisper base fp16 | faster-whisper large-v3-turbo **int8** or mlx-whisper turbo |

**macOS 추천 조합**
1. **기본**: faster-whisper `base` + `int8` — 실습 코드 그대로, 한국어 파인튜닝 체크포인트도 로드 가능.
2. **고속**: mlx-whisper `large-v3-turbo` — 통합메모리 활용, 빠르고 설치 간단.
3. **초경량/온디바이스**: SenseVoice GGUF q8 — 리치 전사(감정/이벤트)가 필요할 때.
4. **API 병행**: `gpt-4o-transcribe`(2-L) — 로컬 모델 없이 바로 실측 지연/품질.

## 1-12. 환경·설치 원칙 (전 세션 공통 · 2-M·2-R 교훈)

1. **설치 셀이 모든 import보다 먼저** — 이미 로드된 C 확장(torch/numpy) 위에 pip로 새 버전을 깔면 ABI가 어긋나 재시작 루프.
2. **"재시작 증상" 패턴으로 판단** — 예외 타입이 아니라 "재시작하라"는 증상 문구(`numpy.dtype size changed`, `undefined symbol`)가 보이면 런타임 재시작 후 ①부터.
3. **T4는 bf16 미지원** → `dtype=torch.float16` 명시 (2-M), 증상 시 fp32 폴백.
4. **한글 폰트** (NanumGothic) — 그래프 한글 깨짐 방지. 설치 후에도 □□면 세션 재시작.
5. **세션은 휘발성** — 공용 함수는 매번 복원하거나 `.py` 모듈로 저장 (`asr_module.py` → 5주차에서 import).
6. **GPU 반납 패턴** — `del 변수`(호출자 스코프에서!) → `free_gpu()` (2-W 교훈).


# 2. 실험에 필요한 함수/클래스 정의 및 주석 🔧

> **✅ = 실행 가능한 코드 셀** (GPU·모델 불필요, numpy/re만 필요) · **📄 = 시그니처+설명 요약** (모델 필요)
> 코드는 위에서 아래로 실행하세요. 각 셀은 자가 점검(assert)을 포함합니다.

| 번호 | 항목 | 실행 |
|---|---|---|
| 2.0 | 공용 유틸: `normalize_ko`·`levenshtein`·`cer`·`add_noise_snr`·`measure_snr` | ✅ |
| 2.1 | 데이터 계약: `CONTRACT_KEYS`·`validate_contract`·`STREAM_EVENT_KEYS` | ✅ |
| 2.2 | 신뢰도·재시도: `judge_confidence`·`should_retry` | ✅ |
| 2.3 | 환각 필터 + 후처리: `is_hallucination`·`postprocess_ko` | ✅ |
| 2.4 | VAD 발화 분리: `split_utterances`·`speech_ratio` | ✅ |
| 2.5 | 스트리밍: `stream_chunks`·`energy_vad`·`find_endpoint`·`local_agreement_commit` | ✅ |
| 2.6 | DER 채점: `der` | ✅ |
| 2.7 | full-duplex FSM: `FullDuplexPolicy` | ✅ |
| 2.8 | 어댑터+팩토리: `ASREngine`·`OpenAIWhisperEngine`·`FasterWhisperEngine`·`create_engine` | ✅ (Mock 검증) |
| 2.9 | 벤치 하네스: `run_benchmark` | ✅ (Mock 검증) |
| 2.10 | 설정: `ASRConfig` | ✅ |
| 2.11 | macOS 어댑터: `mac_fw_transcribe`·`mac_mlx_transcribe` | ✅ |
| 2.12 | 프로덕션 모듈: `preprocess_audio`·`transcribe_safe`·`KoreanASR` | 📄 |
| 2.13 | DSP 밑바닥 (2-F) | 📄 (코드는 원본 노트북) |
| 2.14 | 다이어라이제이션 파이프라인 (2-D) | 📄 |
| 2.15 | 엔진별 어댑터 (2-S/2-V/2-W/2-C) | 📄 |


In [ ]:
# ═══ 2.0 공용 유틸 — 평가·노이즈 (모든 세션 공통) ═══
# ▶ 읽는 순서: normalize_ko(문자 정리) → levenshtein(편집 거리) → cer(오류율)
#   add_noise_snr / measure_snr 는 '잡음 환경 실험'을 위한 쌍 — 만든 잡음이 의도대로인지 되묻는다.
import re
import numpy as np

def normalize_ko(text):
    """한국어 CER용 정규화: 공백·문장부호 제거, 소문자화 (2-1/2-3).
    띄어쓰기만 다른 것('배송조회' vs '배송 조회')은 감점하지 않기 위함."""
    return re.sub(r"[\s\.,\?\!~\-\'\"\u2018\u2019\u201c\u201d]", "", text or "").lower()

def levenshtein(a, b):
    """최소 편집 거리(S+D+I) — 동적 계획법. CER의 근간 (2-1)."""
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def cer(ref, hyp):
    """글자 단위 오류율 (S+D+I)/N — 한국어 ASR 표준 지표 (2-1).
    한국어는 띄어쓰기가 유동적이라 WER 대신 CER을 쓴다."""
    r, h = normalize_ko(ref), normalize_ko(hyp)
    if not r:
        return 0.0 if not h else 1.0
    return levenshtein(r, h) / len(r)

def add_noise_snr(clean, snr_db, seed=42):
    """원하는 SNR(dB)이 되도록 백색소음을 섞는다 (전 세션 가혹조건 공용).
    SNR이 낮을수록 CER이 무너지는 곡선(2-1)을 그릴 때 사용."""
    rng = np.random.default_rng(seed)
    rms_s = np.sqrt(np.mean(clean ** 2))
    noise = rng.normal(0, 1, len(clean)).astype(np.float32)
    noise *= rms_s / (10 ** (snr_db / 20)) / (np.sqrt(np.mean(noise ** 2)) + 1e-12)
    mixed = clean + noise
    return mixed / (np.max(np.abs(mixed)) + 1e-12) * 0.95   # 클리핑 방지

def measure_snr(clean, noisy):
    """실측 SNR(dB) — 노이즈 실험이 의도대로 됐는지 검증 (2-C 등)."""
    noise = noisy - clean
    p_s, p_n = np.mean(clean ** 2), np.mean(noise ** 2) + 1e-12
    return float(10 * np.log10(p_s / p_n))

# ── 자가 점검 (모델·GPU 불필요) ──
assert cer("환불 요청합니다", "환불 요청합니다") == 0.0
assert cer("환불", "환뿔") == 0.5
assert abs(cer("배송 조회 부탁해요", "배송 조희 부탁해") - 0.25) < 1e-9   # 2-1 퀴즈 Q3

# measure_snr 검증: 정규화 없이 직접 구성한 신호로 정확히 5dB
x = (np.random.randn(16000) * 0.1).astype(np.float32)
noise = (np.random.randn(16000) * 0.1 / (10 ** (5 / 20))).astype(np.float32)
y = x + noise
assert abs(measure_snr(x, y) - 5) < 1.0

# add_noise_snr 검증: 피크 정규화(0.95) 후에도 길이·범위가 유효
# ⚠️ add_noise_snr는 0.95 피크 정규화를 거치므로 실측 SNR은 목표에서 어긋날 수 있음(원본 동일 특성)
y2 = add_noise_snr(x, 5)
assert y2.shape == x.shape and np.max(np.abs(y2)) <= 1.0
print("공용 유틸 로드 완료 ✅ (CER·노이즈 검증 통과)")


In [ ]:
# ▶ 데모 1 — CER 계산을 눈으로 따라가기 (초보자용)
# 2.0의 cer()가 내부적으로 하는 일을 3단계로 펼쳐 보면 점수가 어떻게 나오는지 보인다.

ref = "환불 요청합니다"
hyp = "환불 요청했어요"

# ① 정규화: 띄어쓰기·문장부호 제거, 소문자화
print("① 정규화:", repr(normalize_ko(ref)), "vs", repr(normalize_ko(hyp)))

# ② 편집 거리: 몇 글자를 바꿔야 같아지는가
d = levenshtein(normalize_ko(ref), normalize_ko(hyp))
print(f"② 편집 거리: {d} (합니다→했어요: 3글자 치환)")

# ③ 오류율 = 편집 거리 / 참조 글자 수
import numpy as np
n = len(normalize_ko(ref))
print(f"③ CER = {d}/{n} = {d/n:.2f}")
assert cer(ref, hyp) == d / n
print("데모 1 통과 ✅ — CER은 '얼마나 틀렸나'를 글자 단위로 재는 자"  )


In [ ]:
# ═══ 2.1 데이터 계약 — 엔진이 모두 지켜야 하는 최소 형식 (1-4/2-2/2-3) ═══
# ▶ 핵심 사상: '계약은 최소 보장, 확장은 자유'. 필수 키만 있으면 어떤 엔진이든 OK.
#   validate_contract는 bool 반환(조용한 실패) — '시끄러운 실패(예외)'를 원하면 2.8 어댑터에서 assert로 강화한다.
CONTRACT_KEYS = {"utt_id", "engine", "text", "language",
                 "confidence_ok", "avg_logprob", "latency_ms"}

def validate_contract(rec):
    """계약 준수 여부: 필수 키만 존재하면 OK.
    추가 키(emotion/events 등 확장 필드)는 자유 — '계약은 최소 보장, 확장은 자유' (2-V)."""
    return isinstance(rec, dict) and set(rec.keys()) >= CONTRACT_KEYS

# 스트리밍 계약 (2-R): 배치 계약의 '상위 호환' — is_final=True 이벤트는 배치 키도 포함
STREAM_EVENT_KEYS = {"event", "text", "is_final", "chunk_index",
                     "latency_ms", "audio_ms_processed"}

# ── 자가 점검 ──
assert validate_contract({k: None for k in CONTRACT_KEYS})
assert not validate_contract({"text": "누락 있음"})
print(f"데이터 계약 정의 완료 ✅ (필수 키 {len(CONTRACT_KEYS)}개 + 확장 자유)")


In [ ]:
# ═══ 2.2 신뢰도 판정·재시도 정책 — 2-1 asr_v3의 정책 객체화 (2-3) ═══
# ▶ 두 함수가 정책을 객체화한다: judge_confidence(신뢰 판정) + should_retry(재시도 여부).
#   'API형 엔진은 logprob을 안 준다'는 2-C 관찰 — avg_logprob=None이면 텍스트 존재만으로 신뢰(정책이 엔진 차이 흡수).
LOGPROB_THRESHOLD = -0.8   # 2-1 실험으로 정한 기준선 (엔진·도메인별 튜닝 대상)

def judge_confidence(avg_logprob, text, threshold=LOGPROB_THRESHOLD):
    """전사 결과를 신뢰할 수 있는가?
    ① 빈 텍스트 → False
    ② logprob 없음(API형 엔진: Cohere/Qwen/Omnilingual) → 텍스트 존재만으로 True
    ③ logprob < 임계값 → False
    정책 함수가 엔진 차이(2-C에서 본 'API는 logprob을 안 줌')를 흡수해야 어댑터가 성립한다."""
    if not text or not text.strip():
        return False
    if avg_logprob is None:
        return True
    return avg_logprob >= threshold

def should_retry(attempt, confidence_ok, max_retries=1):
    """재시도 여부: 신뢰도 미달 & 남은 횟수 있음.
    2단 전략: 평소 greedy(빠름), 신뢰도 의심될 때만 beam=5 재시도(확실한 길)."""
    return (not confidence_ok) and (attempt < max_retries)

# ── 자가 점검 ──
assert judge_confidence(-0.3, "안녕하세요") is True
assert judge_confidence(-1.2, "안녕하세요") is False
assert judge_confidence(None, "안녕하세요") is True
assert should_retry(0, False) and not should_retry(1, False)
print("신뢰도/재시도 정책 검증 ✅")


In [ ]:
# ═══ 2.3 환각 필터 + 한국어 후처리 (2-3) ═══
# ▶ 방어 장치 두 개: is_hallucination(지어낸 문장 차단) + postprocess_ko(오인식 교정).
#   환각은 단독 신호로 삭제하지 말 것 — '네 네 네 네' 정상 반복 오차단 방지가 주석에 명시됨.
import re

HALLUCINATION_PATTERNS = [   # Whisper 계열이 무음·잡음에서 자주 짓는 유튜브식 문구
    "시청해주셔서 감사합니다", "구독", "좋아요", "MBC 뉴스",
    "자막 제공", "다음 영상에서 만나요", "Thank you for watching",
]

def is_hallucination(text, audio_duration_s, speech_ratio):
    """환각 의심 판정 — 세 신호로 잡는다:
    ① 말 비율(speech_ratio) 5% 미만인데 텍스트가 있으면 의심
    ② 정형 문구 포함 ③ 같은 어절 4회 연속 반복(루프 환각)
    ⚠️ '네 네 네 네' 같은 정상 반복을 오차단할 수 있어 단독 신호로 삭제하지 말 것."""
    t = (text or "").strip()
    if not t:
        return False
    if speech_ratio < 0.05:
        return True
    for p in HALLUCINATION_PATTERNS:
        if p in t:
            return True
    words = t.split()
    run = 1
    for i in range(1, len(words)):
        run = run + 1 if words[i] == words[i - 1] else 1
        if run >= 4:
            return True
    return False

DOMAIN_LEXICON = {   # 콜센터 관측 오인식 → 표준 표기 (2-3 / 2-V ITN의 결정적 미니 버전)
    "환뿔": "환불", "에이 에스": "AS", "에이에스": "AS",
    "와이 파이": "와이파이", "카드 사": "카드사",
}
_NUM_UNIT_RE = re.compile(r"(\d[\d,]*)\s+(원|일|개|번|시|분|명)")

def postprocess_ko(text, lexicon=None):
    """도메인 교정 → 숫자-단위 붙여쓰기('3 일'→'3일') → 공백 정리.
    initial_prompt(전사 전, 확률적)와 달리 이건 전사 후 결정적 교정 — 실전은 둘 다 쓴다."""
    lexicon = lexicon or DOMAIN_LEXICON
    out = text or ""
    for wrong, right in lexicon.items():
        out = out.replace(wrong, right)
    out = _NUM_UNIT_RE.sub(r"\1\2", out)
    return re.sub(r"\s{2,}", " ", out).strip()

# ── 자가 점검 ──
assert is_hallucination("시청해주셔서 감사합니다", 3.0, 0.5) is True
assert is_hallucination("네 네 네 네 확인했습니다", 3.0, 0.5) is True
assert is_hallucination("배송이 어디까지 왔나요", 3.0, 0.01) is True
assert is_hallucination("배송이 어디까지 왔나요", 3.0, 0.5) is False
assert postprocess_ko("환뿔 접수 부탁드립니다") == "환불 접수 부탁드립니다"
assert postprocess_ko("3 일 이내에 30,000 원 환불됩니다") == "3일 이내에 30,000원 환불됩니다"
print("환각 필터·후처리 검증 ✅")


In [ ]:
# ═══ 2.4 VAD 발화 분리 (1-3 에너지 VAD 계승 — 2-3/2-D 공용) ═══
# ▶ split_utterances: 연속 오디오 → (시작,끝) 발화 목록. 에너지 VAD + 침묵 카운터(EoU).
#   min_silence_s가 '말을 끊는 에이전트(↓) vs 굼뜬 에이전트(↑)' 트레이드오프 축 (1-3).
import numpy as np

def split_utterances(audio, sr, frame_ms=30, energy_thresh=0.02, min_silence_s=0.8):
    """연속 오디오 → (시작, 끝) 발화 구간 목록.
    에너지 VAD + EoU(침묵 카운터). min_silence_s가
    '말 끊는 에이전트(↓) vs 굼뜬 에이전트(↑)' 트레이드오프의 축 (1-3)."""
    frame_len = int(sr * frame_ms / 1000)
    min_silence_frames = int(min_silence_s * 1000 / frame_ms)
    segments, start, silence = [], None, 0
    n_frames = len(audio) // frame_len
    for i in range(n_frames):
        frame = audio[i * frame_len:(i + 1) * frame_len]
        rms = float(np.sqrt(np.mean(frame.astype(np.float64) ** 2)))
        if rms > energy_thresh:
            if start is None:
                start = i * frame_len
            silence = 0
        elif start is not None:
            silence += 1
            if silence >= min_silence_frames:
                segments.append((start, (i + 1 - silence) * frame_len))
                start, silence = None, 0
    if start is not None:
        segments.append((start, n_frames * frame_len))
    return segments

def speech_ratio(audio, sr, frame_ms=30, energy_thresh=0.02):
    """전체 중 말 프레임 비율 — 환각 필터의 ①번 신호 입력 (2-3)."""
    frame_len = int(sr * frame_ms / 1000)
    n_frames = len(audio) // frame_len
    if n_frames == 0:
        return 0.0
    speaking = sum(
        1 for i in range(n_frames)
        if float(np.sqrt(np.mean(audio[i*frame_len:(i+1)*frame_len].astype(np.float64) ** 2)))
        > energy_thresh)
    return speaking / n_frames

# ── 자가 점검 (합성 스트림: 발화 1초 + 침묵 1초) × 2 ──
t = np.arange(16000) / 16000
burst = (np.sin(2 * np.pi * 300 * t) * 0.5).astype(np.float32)
sil = np.zeros(16000, dtype=np.float32)
stream = np.concatenate([burst, sil, burst, sil])
assert len(split_utterances(stream, 16000)) == 2
assert len(split_utterances(stream, 16000, min_silence_s=1.5)) == 1  # 굼뜬 EoU → 1발화
print("VAD 발화 분리 검증 ✅ (min_silence 0.8s→2발화 / 1.5s→1발화)")


In [ ]:
# ▶ 데모 2 — VAD가 '발화 1초 + 침묵 1초'에서 발화 2개를 찾는 과정
import numpy as np

sr = 16000
t = np.arange(sr) / sr
burst = (np.sin(2 * np.pi * 300 * t) * 0.5).astype(np.float32)   # 1초짜리 '말소리'
sil = np.zeros(sr, dtype=np.float32)                              # 1초짜리 '침묵'
stream = np.concatenate([burst, sil, burst, sil])                 # 말-침묵-말-침묵

segs = split_utterances(stream, sr)                               # 2.4 함수
print("발화 구간 (시작, 끝) ms:", [(s, e) for s, e in segs])
print("음성 비율:", round(speech_ratio(stream, sr), 2))
assert len(segs) == 2                                             # 발화는 정확히 2개
assert 0.4 < speech_ratio(stream, sr) < 0.6                       # 4초 중 ~2초가 말소리
print("데모 2 통과 ✅ — VAD는 '소리의 문단'을 찾아내는 분리자")


In [ ]:
# ═══ 2.5 스트리밍 유틸 (2-R) — 청크·VAD·엔드포인트·LocalAgreement ═══
# ▶ 4개 유틸이 스트리밍 파이프라인을 이룬다: stream_chunks(자르기) → energy_vad(음성?) →
#   find_endpoint(끝점) → local_agreement_commit(흔들리는 가설 확정 규칙).
#   local_agreement: '요금'→'요금제' 흔들림에서 이미 확정한 접두어는 절대 뒤집히지 않게 한다.
import time
import numpy as np

def stream_chunks(audio, sr=16000, chunk_ms=320, realtime=False):
    """오디오를 chunk_ms 단위로 잘라 순서대로 내보내는 제너레이터.
    Colab에는 마이크가 없어서 완성된 wav를 흘려보내 실시간 입력을 재현한다 (2-R)."""
    chunk_size = int(sr * chunk_ms / 1000)
    for i in range(0, len(audio), chunk_size):
        chunk = audio[i:i + chunk_size]
        if realtime:
            time.sleep(len(chunk) / sr)   # 실제 스트림 속도로 페이싱
        yield chunk

def energy_vad(chunk, threshold_db=-35.0):
    """청크 RMS가 threshold_db(dBFS) 이상이면 음성 판정.
    ⚠️ 반환을 파이썬 bool로 캐스팅 — np.bool_는 json.dumps를 거부한다 (2-R 교훈)."""
    if len(chunk) == 0:
        return False
    rms = float(np.sqrt(np.mean(chunk.astype(np.float64) ** 2)))
    return bool(20.0 * np.log10(rms + 1e-10) > threshold_db)

def find_endpoint(vad_flags, silence_chunks=3):
    """음성 시작 후 연속 무음 silence_chunks개가 처음 완성되는 청크 인덱스.
    3청크 × 320ms = 960ms — AICC 관행(0.7~1.0초). 너무 짧으면 말을 끊고 길면 예산을 먹는다."""
    started, run = False, 0
    for i, is_sp in enumerate(vad_flags):
        if is_sp:
            started, run = True, 0
        elif started:
            run += 1
            if run >= silence_chunks:
                return i
    return None

def local_agreement_commit(prev_tokens, curr_tokens, committed):
    """연속 두 가설이 일치하는 접두어만 확정 → 확정분은 절대 뒤집히지 않는다.
    (6주차에서 확정 텍스트는 즉시 LLM→TTS로 가므로 '오확정 금지'가 핵심.)
    반환: (새 committed 리스트, 이번에 새로 확정된 토큰 리스트)"""
    n = min(len(prev_tokens), len(curr_tokens))
    agree = 0
    while agree < n and prev_tokens[agree] == curr_tokens[agree]:
        agree += 1
    if agree <= len(committed):
        return list(committed), []
    newly = curr_tokens[len(committed):agree]
    return list(committed) + newly, newly

# ── 자가 점검: '요금'→'요금제' 흔들림이 확정에서 차단되는지 (2-R 소스 트레이스) ──
committed, prev = [], []
for hyp in [
    ["네", "고객님", "요금"],
    ["네", "고객님", "요금제", "변경"],
    ["네", "고객님", "요금제", "변경", "도와"],
]:
    committed, newly = local_agreement_commit(prev, hyp, committed)
    prev = hyp
    print("가설:", " ".join(hyp), "→ 확정:", " ".join(committed),
          f"(+{' '.join(newly) if newly else '없음'})")
print("LocalAgreement-2 검증 ✅ (요금은 확정 안 됨 = 오확정 방지)")


In [ ]:
# ═══ 2.6 다이어라이제이션 채점 — DER (2-D) ═══
# ▶ der(): 화자 구분 정확도. 클러스터 번호는 임의(0/1 뒤집혀도 정답) → 최적 매핑을 시도.
#   miss(놓침) + false_alarm(과감지) + confusion(화자 혼동) 을 참조 발화 시간으로 나눈다.
import numpy as np
from itertools import permutations

NOSPK = -1   # 비발화 라벨

def der(ref_segments, hyp_segments, total_ms, frame_ms=100):
    """DER = (miss + false_alarm + confusion) / 참조 발화 시간.
    클러스터 번호는 임의(0/1이 뒤집혀도 정답) → 모든 화자 매핑을 시도해
    confusion이 최소인 매핑으로 채점한다.
    segments: [(start_ms, end_ms, speaker_id), ...]"""
    def to_labels(segments, n):
        labels = np.full(n, NOSPK, dtype=int)
        for s, e, spk in segments:
            labels[int(round(s / frame_ms)):int(round(e / frame_ms))] = spk
        return labels

    n = int(np.ceil(total_ms / frame_ms))
    ref, hyp = to_labels(ref_segments, n), to_labels(hyp_segments, n)
    ref_sp, hyp_sp = ref != NOSPK, hyp != NOSPK
    n_ref = int(ref_sp.sum())

    miss = int((ref_sp & ~hyp_sp).sum())     # VAD가 보수적 → 발화를 놓침
    fa = int((~ref_sp & hyp_sp).sum())       # VAD가 민감·잡음 → 비발화를 발화로
    both = ref_sp & hyp_sp

    ref_ids = sorted(set(ref[ref_sp].tolist()))
    hyp_ids = sorted(set(hyp[hyp_sp].tolist()))
    best_conf = None
    for perm in set(permutations(ref_ids + [None] * max(0, len(hyp_ids) - len(ref_ids)),
                                len(hyp_ids))):
        mapping = dict(zip(hyp_ids, perm))
        conf = int(sum(1 for i in np.where(both)[0] if mapping.get(int(hyp[i])) != ref[i]))
        best_conf = conf if best_conf is None else min(best_conf, conf)
    return {"der": round((miss + fa + best_conf) / n_ref, 4),
            "miss": round(miss / n_ref, 4),
            "false_alarm": round(fa / n_ref, 4),
            "confusion": round(best_conf / n_ref, 4)}

# ── 자가 점검: 화자 번호가 뒤집혀도 DER 0 ──
_r = [(0, 1000, 0), (1500, 2500, 1)]
res = der(_r, [(0, 1000, 1), (1500, 2500, 0)], 3000)
assert res["der"] == 0.0, res
print("DER 검증 ✅ (라벨이 뒤집혀도 0 — 최적 매핑 덕분)")


In [ ]:
# ═══ 2.7 full-duplex 미니 정책 FSM (2-L) — 매 프레임(40ms) 의사결정 ═══
# ▶ FullDuplexPolicy.step(user, agent) 가 40ms 프레임마다 4가지 행동 중 하나를 고른다.
#   R1 YIELD(끼어들기 수용) / R2 BACKCHANNEL(맞장구) / R3 SPEAK(침묵 후 응답) / R4 LISTEN.
FRAME_MS = 40

class FullDuplexPolicy:
    LISTEN, BACKCHANNEL, SPEAK, YIELD = "LISTEN", "BACKCHANNEL", "SPEAK", "YIELD"

    def __init__(self, eou_threshold_ms=240, backchannel_every_ms=2000):
        self.eou_threshold_ms = eou_threshold_ms          # R3: 사용자 침묵 누적 → SPEAK
        self.backchannel_every_ms = backchannel_every_ms  # R2: 맞장구 주기
        self.user_talk_accum_ms = 0    # 사용자 연속 발화 누적
        self.silence_accum_ms = 0      # 사용자 침묵 누적
        self.last_backchannel_at = 0

    def step(self, user_speaking, agent_speaking):
        """규칙 4개 — 실제 GPT-Live는 신경망이지만 '매 프레임 상태 기반 판단' 뼈대는 동일:
        R1 사용자 발화 중 AND 에이전트 발화 중 → YIELD (사용자 우선, barge-in 수용)
        R2 사용자 연속 발화 2초마다       → BACKCHANNEL (맞장구)
        R3 사용자 침묵 누적 ≥ 임계값       → SPEAK (응답 시작)
        R4 그 외                         → LISTEN"""
        if user_speaking:
            self.user_talk_accum_ms += FRAME_MS
            self.silence_accum_ms = 0
            if agent_speaking:
                return self.YIELD
            if (self.user_talk_accum_ms - self.last_backchannel_at
                    >= self.backchannel_every_ms):
                self.last_backchannel_at = self.user_talk_accum_ms
                return self.BACKCHANNEL
            return self.LISTEN
        self.silence_accum_ms += FRAME_MS
        return self.SPEAK if self.silence_accum_ms >= self.eou_threshold_ms else self.LISTEN

def run_policy_on_pattern(pattern, **kw):
    """프레임별 (user_speaking, agent_speaking) 리스트 → 행동 시퀀스."""
    policy = FullDuplexPolicy(**kw)
    return [policy.step(u, a) for (u, a) in pattern]

# ── 시나리오 A: 사용자 400ms(10프레임) 발화 후 침묵 → 침묵 240ms 누적 시 SPEAK ──
pattern = [(True, False)] * 10 + [(False, False)] * 10
acts = run_policy_on_pattern(pattern)
spk_at = acts.index("SPEAK") + 1
print(f"시나리오 A: SPEAK 지점 = {spk_at}프레임 ({spk_at * FRAME_MS}ms) — 캐스케이드 EoU 700ms보다 빠름")
print("행동 시퀀스:", "→".join(acts))


In [ ]:
# ═══ 2.8 어댑터 + 팩토리 — 어떤 엔진이든 표준 계약으로 (2-2) ═══
# ▶ BaseASR(추상 인터페이스) + AdapterFactory(등록부) — 엔진별 어댑터가 표준 계약으로 접힌다.
#   '엔진 모양을 어댑터가 계약으로 접는다': 이 사상이 시스템 통합 노트의 어댑터 ABC로 이어진다.
import time
import numpy as np

class ASREngine:
    """표준 계약을 지키는 ASR 어댑터의 공통 틀.
    엔진별로 `_raw_transcribe`만 구현하면 `transcribe`(표준 계약 반환)는 공유한다.
    여행용 어댑터처럼 제각각인 엔진 인터페이스를 우리 계약으로 변환하는 껍데기."""
    name = "base-engine"

    def _raw_transcribe(self, path):
        """(텍스트, avg_logprob) 반환 — 엔진별 구현부."""
        raise NotImplementedError

    def transcribe(self, path, logprob_th=-1.0):
        """표준 계약: 어떤 엔진이든 이 형식으로 반환 (2-1 asr_v3 계승)."""
        t0 = time.perf_counter()
        text, lp = self._raw_transcribe(path)
        ok = (lp >= logprob_th) and len(text.strip()) > 0
        return {"engine": self.name, "text": text, "confidence_ok": ok,
                "avg_logprob": round(lp, 3),
                "latency_ms": round((time.perf_counter() - t0) * 1000)}

# 실전 엔진 2종 — 생성 시점에만 모델 로드 (정의만으로는 GPU·패키지 불필요)
class OpenAIWhisperEngine(ASREngine):
    def __init__(self, size="base", device="cuda"):
        import whisper, torch
        self.name = f"openai-{size}"
        self.device = device if torch.cuda.is_available() else "cpu"
        self.model = whisper.load_model(size, device=self.device)
    def _raw_transcribe(self, path):
        out = self.model.transcribe(path, language="ko", fp16=(self.device == "cuda"))
        lps = [s.get("avg_logprob", 0.0) for s in out.get("segments", [])]
        return out["text"], (float(np.mean(lps)) if lps else -10.0)

class FasterWhisperEngine(ASREngine):
    def __init__(self, size="base", compute_type="float16", device="cuda"):
        from faster_whisper import WhisperModel
        self.name = f"faster-{size}-{compute_type}"
        self.model = WhisperModel(size, device=device, compute_type=compute_type)
    def _raw_transcribe(self, path):
        segments, _ = self.model.transcribe(path, language="ko", beam_size=1)
        segs = list(segments)
        lps = [s.avg_logprob for s in segs]
        return "".join(s.text for s in segs), (float(np.mean(lps)) if lps else -10.0)

def create_engine(config):
    """팩토리: config['asr'] 한 줄이 엔진을 결정 (1-5 config.json 구조와 동일).
    교체 지점을 한 곳으로 모아 에이전트 코드는 아무것도 고칠 필요가 없다."""
    kind = config["asr"]["engine"]
    if kind == "openai-whisper":
        return OpenAIWhisperEngine(config["asr"].get("model", "base"))
    if kind == "faster-whisper":
        return FasterWhisperEngine(config["asr"].get("model", "base"),
                                   config["asr"].get("compute_type", "float16"))
    raise ValueError(f"모르는 엔진: {kind}")

# ── 계약 검증: 모델 없이 Mock 엔진으로 표준 계약이 성립하는지 확인 ──
class MockEngine(ASREngine):
    name = "mock"
    def __init__(self, text="배송 조회 도와드리겠습니다", lp=-0.3):
        self._text, self._lp = text, lp
    def _raw_transcribe(self, path):
        return self._text, self._lp

rec = MockEngine(lp=-1.2).transcribe("any.wav")   # logprob 낮음 → confidence_ok False
assert rec["confidence_ok"] is False
assert isinstance(rec["avg_logprob"], float)
print("어댑터 계약 검증 ✅ → MockEngine ok=", rec["confidence_ok"], "| text=", rec["text"])
print("💡 엔진 교체 = config['asr']['engine'] 한 줄만 변경 (에이전트는 무감동)")


In [ ]:
# ═══ 2.9 벤치 하네스 — 어떤 엔진이든 같은 잣대로 심사 (2-2) ═══
# ▶ BenchHarness: 어떤 엔진이든 같은 잣대(CER·지연·재시도)로 심사하는 '심사대'.
#   공정한 벤치마크 = 동일 데이터 + 동일 계약 + 동일 하드웨어 (1-5 함정 방지).
# 의존: 2.0의 cer(), numpy, time — 위 셀을 먼저 실행할 것

def run_benchmark(engine_name, transcribe_fn, eval_set, verbose=True):
    """transcribe_fn: 파일경로 → 인식텍스트. eval_set: [(path, ref, cond, dur), ...]
    cond는 'clean'/'noisy'. 반환: clean/noisy CER·평균 지연·RTF (성적표 한 행).
    엔진이 바뀌어도 이 함수는 그대로 — 시험지·심판은 고정, 선수만 바뀐다."""
    rows = []
    for path, ref, cond, dur in eval_set:
        t0 = time.perf_counter()
        hyp = transcribe_fn(path)
        latency = time.perf_counter() - t0
        rows.append({"cond": cond, "cer": cer(ref, hyp),
                     "latency": latency, "rtf": latency / dur})
    result = {"engine": engine_name,
              "cer_clean": np.mean([r["cer"] for r in rows if r["cond"] == "clean"]),
              "cer_noisy": np.mean([r["cer"] for r in rows if r["cond"] == "noisy"]),
              "latency_avg": np.mean([r["latency"] for r in rows]),
              "rtf_avg": np.mean([r["rtf"] for r in rows])}
    if verbose:
        print(f"[{engine_name}] clean CER {result['cer_clean']:.1%} | "
              f"noisy CER {result['cer_noisy']:.1%} | "
              f"평균 지연 {result['latency_avg']:.2f}s | RTF {result['rtf_avg']:.2f}")
    return result

# ── Mock 검증: 예산 0.5s 기준 우승자 판정 절차를 그대로 체험 ──
fake_eval = [("a.wav", "안녕하세요", "clean", 1.0),
             ("b.wav", "환불 도와드리겠습니다", "noisy", 1.0)]
def mock_fn(path):
    return "안녕하세요" if path.startswith("a") else "환불 도와드리겠습니다"
r = run_benchmark("mock", mock_fn, fake_eval)
print("💡 실전: 시험지(clean+noisy 10문항) 생성 → 엔진별 transcribe_fn 등록 →")
print("   예산 0.5s 통과 엔진 중 min(noisy CER) = 우승 (2-2 토너먼트 규정)")


In [ ]:
# ═══ 2.10 ASRConfig — 정책 숫자를 코드에서 분리 (2-3) ═══
# ▶ ASRConfig: 정책 숫자(임계값·타임아웃)를 코드에서 분리해 JSON으로 관리.
#   '숫자를 하드코딩하면 실험마다 바꾸기 어렵다' → 설정 객체로 빼는 패턴.
import json
from dataclasses import dataclass, asdict

@dataclass
class ASRConfig:
    engine_name: str = "faster-whisper-base"
    model_size: str = "base"            # T4에서 500ms 예산을 지키는 크기 (2-2 실측 근거)
    device: str = "cuda"
    compute_type: str = "float16"       # 2-2: T4에서 정확도 손실 거의 없이 2배 빠름
    language: str = "ko"
    beam_size: int = 1                  # greedy — 재시도 때만 beam 확장 (2-1 트레이드오프)
    retry_beam_size: int = 5
    logprob_threshold: float = -0.8     # 신뢰도 기준선 (엔진·도메인별 튜닝 대상)
    max_retries: int = 1
    vad_frame_ms: int = 30              # 1-3 VAD 프레임
    vad_energy_thresh: float = 0.02
    min_silence_s: float = 0.8          # '말 끊는 vs 굼뜬' 트레이드오프 균형점 (1-3)
    latency_budget_ms: float = 500.0    # 1-3 지연 예산 중 ASR 몫
    initial_prompt: str = "콜센터 고객 문의 전화입니다. 배송, 환불, 상담원 연결 관련 내용입니다."

CFG = ASRConfig()
print(json.dumps(asdict(CFG), ensure_ascii=False, indent=2))


In [ ]:
# ═══ 2.11 macOS(Apple Silicon) 실행 어댑터 — Colab과의 차이는 설정 한 줄 🍎 ═══
# ▶ Apple Silicon에서 GPU(Colab T4) 없이도 실행되게 하는 어댑터.
#   Colab과의 차이는 '설정 한 줄' — 같은 계약을 지키므로 나머지 코드는 그대로 재사용.
# T4('cuda'/'float16') → Mac('cpu'/'int8') 로만 바꾸면 기존 실습 코드 재사용 가능.

def mac_fw_transcribe(model, audio_path, language="ko", beam_size=5):
    """faster-whisper macOS — Colab과 동일 API.
    로드 예: WhisperModel('large-v3-turbo', device='cpu', compute_type='int8')  # 약 1.5GB
    한국어 파인튜닝 체크포인트(ghost613/...-korean)도 CT2라 같은 방식으로 로드 가능."""
    segments, info = model.transcribe(audio_path, language=language, beam_size=beam_size)
    return "".join(s.text for s in segments)

def mac_mlx_transcribe(model_path, audio_path, language="ko"):
    """mlx-whisper (Apple MLX) — M시리즈 통합 메모리 최적화, 설치: pip install mlx-whisper.
    model_path: HF 리포명 예) 'mlx-community/whisper-large-v3-turbo' (int4/int8 양자화본)."""
    import mlx_whisper          # 호출 시점에만 import — 정의만으로는 무설치
    result = mlx_whisper.transcribe(audio_path, path_or_hf_repo=model_path, language=language)
    return result["text"]

# whisper.cpp(Metal/Core ML)는 CLI 위주 — 실행 예시:
#   ./main -m models/ggml-large-v3-turbo.bin -l ko -f call.wav
# SenseVoice GGUF(q8≈254MB): llama.cpp 계열 바이너리로 온디바이스 실행 (2-V 참고)

print("macOS 어댑터 정의 완료 ✅ (import는 호출 시점에만)")
print("예시: model = WhisperModel('large-v3-turbo', device='cpu', compute_type='int8')")


## 2.12 📄 프로덕션 ASR 모듈 요약 (2-3) — GPU 필요 → 시그니처·역할만

> 원본 코드는 `ASR/2-3_음성인식구현_실습.ipynb` — 오늘의 산출물은 `asr_module.py`로 저장되어 5주차(LLM)·6주차(LiveKit)에서 import합니다.

| 함수/클래스 | 시그니처 | 역할 | 요구사항 |
|---|---|---|---|
| `preprocess_audio` | `(path_or_array, sr_in=None) -> (y, dur)` | 어떤 입력이든 **16kHz 모노 float32, 피크 0.95**로 통일 | F1 |
| `transcribe_safe` | `(path_or_array, utt_id, sr_in, cfg) -> 계약 dict` | 전처리→전사→신뢰도→재시도→계약. **예외 시에도 계약 dict 반환** (status 필드로 보고) | F2,F3,NF3 |
| `transcribe_stream` | `(audio, sr, cfg) -> [계약 dict]` | VAD 발화 분리→발화별 전사, `eou_wait_ms`·`t_start_s` 기록 | F5 |
| `KoreanASR` | `recognize() / recognize_stream() / close()` | 2~4장 부품을 클래스로 조립. 생성자에서 로드+워밍업, `recognize`는 환각 필터+후처리 포함 완성본 | F1~F6,NF |
| `ASRConfig` | dataclass (2.10) | 정책 숫자 분리 | — |
| `budget_verdict` | `(latency_ms, budget=500) -> str` | 지연 예산 판정 ('✅ 통과'/'❌ 초과') | NF1 |

**핵심 문장**: 실전 ASR 모듈 = 엔진 + **의심하는 정책**(신뢰도·환각) + **표준 계약** + **시간 감각**(예산).

**요구사항 표(2-3 1-2절) 기억하기** — 기능(F): F1 포맷 통일 / F2 계약 반환 / F3 재시도 1회 / F4 환각 차단 / F5 스트림 분리 / F6 후처리. 비기능(NF): NF1 지연≤500ms / NF2 어댑터 교체 / NF3 예외에도 생존.


## 2.13 📄 DSP 밑바닥 함수 요약 (2-F) — numpy만으로 librosa 재현

> 코드는 `ASR/2-F_파형에서_멜스펙트로그램까지_밑바닥구현.ipynb`. GPU 불필요하므로 원본에서 직접 실행하며 익히세요. Whisper 입력 = `n_fft=400, hop=160, n_mels=80`.

| 함수 | 역할 | 검증 포인트 |
|---|---|---|
| `frame_signal(y, n_fft, hop_length)` | 반사 패딩 + 겹치는 창으로 프레임 행렬 | 가장자리 패딩 규약: **reflect** vs librosa 신버전 기본 constant |
| `hann_window(n_fft)` | w[n]=0.5(1−cos 2πn/N) — **주기형(분모 N)** | 대칭형(N−1)과 값이 미묘하게 달라 검증 실패 |
| `dft_naive(frame)` | 수식 그대로 O(N²) — 원리용 | `np.fft.rfft`와 일치 + 실시간의 수천 배 느림 |
| `stft_manual(y, n_fft, hop)` | 프레이밍+창+FFT 조립 = STFT | 파워 스펙트로그램 = \|X\|² |
| `hz_to_mel / mel_to_hz` | HTK 공식 mel=2595·log₁₀(1+f/700) | 100→200Hz와 7100→7200Hz의 감각 차이 확인 |
| `mel_filterbank(sr, n_fft, n_mels)` | 멜 축 균등 분할→삼각형 필터 (행렬 곱 1번) | (80×201) 형상 |
| `melspectrogram_manual(...)` | 파워 스펙트로그램 × 필터뱅크ᵀ + 로그 압축 | **librosa와 np.allclose로 최종 채점** (htk=True, norm=None, pad_mode='reflect') |
| `power_to_db_manual` | 로그 압축 (log(0) 바닥값 처리) | — |

> **공정한 채점 규약**: 우리가 선택한 규약(HTK·정규화 없음·reflect 패딩)을 librosa에 꼭 알려줘야 `np.allclose`가 통과합니다.


## 2.14 📄 다이어라이제이션 파이프라인 요약 (2-D) — VAD→임베딩→클러스터링

> 코드는 `ASR/2-D_화자분리_다이어라이제이션_실습.ipynb`. GPU는 ECAPA 임베딩(2-D `[3-2]`)에만 필요.

| 함수 | 시그니처 | 역할 |
|---|---|---|
| `energy_vad(y, sr, frame_ms, threshold_ratio)` | → bool 배열 | 프레임 RMS > (최대 RMS × 0.05) → 발화. 1-3 VAD의 10ms판 |
| `cluster_embeddings(embs, n_speakers=2)` | → 라벨 | L2 정규화 후 응집 클러스터링(코사인, 평균 연결) |
| `windows_to_labels(...)` | → 프레임 라벨 | 겹치는 윈도우 라벨 **다수결** + VAD로 비발화 복원 |
| `smooth_labels(labels, min_frames=30)` | → 라벨 | 300ms 미만 침입 런을 이웃에 흡수 |
| `segments_from_labels(labels, frame_ms)` | → [(start,end,spk)] | 런 인코딩 → 세그먼트 |
| `der(...)` (2.6 실행 코드 ✅) | → {der, miss, fa, confusion} | 최적 화자 매핑 포함 채점 |

**실전 주의 (2-D §6)**: 클러스터→실명 매핑은 정답이 있어야 가능. 실전은 ①첫 발화=상담원 규칙 ②등록 음성 유사도(=화자 인식) ③API known speaker 기능으로 이름을 붙임.


## 2.15 📄 엔진별 어댑터 요약 (2-S / 2-V / 2-W / 2-C) — 같은 계약, 다른 규칙

> 각 보충 노트북은 **같은 시험지(utt_001~005 + SNR 5dB)를 같은 계약으로** 치렀습니다. 차이는 전부 어댑터 안에서 흡수됩니다.

| 엔진(노트북) | 어댑터 함수 | 신뢰도 규칙 | 특이사항 |
|---|---|---|---|
| Qwen3-ASR (2-S) | `qwen3_asr_to_contract()` · `fw_transcribe()` | logprob **없음** → `avg_logprob=None, confidence_ok=True` 고정 | LID 실험 `language=None`; fp16 필수(T4 bf16 불가) |
| SenseVoice (2-V) | `sv_transcribe()` · `parse_sensevoice_tags()` · `sensevoice_to_contract()` | 없음 | **확장 필드** emotion/events 추가 — 계약 필수 키는 그대로 |
| Whisper turbo (2-W) | `fw_transcribe()` · `whisper_to_contract()` | **avg_logprob 복원** (임계 -1.0) | `compute_type` 양자화 실험(float16 vs int8_float16), `vram_gb()`로 VRAM 측정 |
| Cohere (2-C) | `cohere_transcribe()` · `basic_engine_to_contract()` | 없음 | 게이티드 HF 인증, `punctuation` 옵션, 35s 초과 자동 청크 |

**핵심 교훈 (2-V)**: "계약은 최소 보장, 확장은 자유" — 필수 키를 유지한 채 emotion/events를 확장 필드로 싣자 기존 하류 파이프라인을 깨지 않고 새 능력이 전달됐다.


## 2.16 [REAL] 실물 실행 — mlx-whisper · faster-whisper 🍎

> 아래 셀은 **실물 모델**을 다운로드해 실제 전사합니다 (1회 수 GB).
> 준비: `bash setup_apple_silicon.sh asr` — 설치된 엔진만 실행되고, 없으면 안내 후 스킵됩니다.
> 목표: 2.8 어댑터·2.9 벤치·2.11 macOS 어댑터를 **실물 출력**으로 흐르게 한다.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd() if (Path.cwd() / "aicc_env.py").exists() else Path.cwd().parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import aicc_env as ae

if not ae.has("mlx_whisper"):
    print("mlx-whisper 미설치 → 스킵.  bash setup_apple_silicon.sh asr")
else:
    import numpy as np
    wav = ae.ensure_test_audio()                       # TTS 생성 or 폴백 wav
    audio, _ = ae.load_wav(wav)
    # 2.11 어댑터: mlx_whisper.transcribe(audio_path, path_or_hf_repo=..., language=ko)
    text, ms = ae.timed(mac_mlx_transcribe, ae.MODEL_CFG["asr_mlx"], str(wav), "ko")
    rec = {"engine": "mlx-whisper:" + ae.MODEL_CFG["asr_mlx"].split("/")[-1],
           "text": text, "confidence_ok": len(text.strip()) > 0,
           "avg_logprob": None, "latency_ms": round(ms, 1)}
    dur_s = audio.size / 16000.0
    print("전사:", text[:90])
    print(f"지연 {ms:.0f}ms | 오디오 {dur_s:.1f}s | RTF {ms / 1000 / dur_s:.2f}x")
    assert rec["confidence_ok"]
    print("mlx-whisper 실물 전사 통과 ✅ (2.11 mac_mlx_transcribe 호출 — 첫 실행은 다운로드 포함)")


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd() if (Path.cwd() / "aicc_env.py").exists() else Path.cwd().parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import aicc_env as ae

if not ae.has("faster_whisper"):
    print("faster-whisper 미설치 → 스킵.  bash setup_apple_silicon.sh asr")
else:
    from faster_whisper import WhisperModel
    model = WhisperModel(ae.MODEL_CFG["asr_faster"], device="cpu",
                         compute_type=ae.MODEL_CFG["asr_faster_compute"])
    wav = ae.ensure_test_audio()
    text_fw, ms = ae.timed(mac_fw_transcribe, model, str(wav), language="ko", beam_size=1)
    if ae.has("mlx_whisper"):
        text_mlx = mac_mlx_transcribe(ae.MODEL_CFG["asr_mlx"], str(wav), "ko")
        print(f"교차 CER(mlx vs faster) = {cer(text_mlx, text_fw):.3f}")
    print("faster-whisper:", text_fw[:90])
    print(f"지연 {ms:.0f}ms")
    assert len(text_fw.strip()) > 0
    print("faster-whisper 실물 교차 검증 통과 ✅ (2.11 mac_fw_transcribe 호출)")


# 3. 실험 진행 방법 🧪

## 3-1. 표준 워크플로우 (모든 노트북 공통 7단계)

```
① 설치  → ② 폰트·시드  → ③ 헬스체크  → ④ 자산 복원(표준 발화·CER·계약)
   → ⑤ 본 실험  → ⑥ 평가·시각화  → ⑦ 산출물 저장 + GPU 반납
```

> ⚠️ **설치 셀은 반드시 첫 실행**. `import torch` 이후 설치하면 재시작 루프(2-M).
> 세션이 바뀌면 디스크가 초기화되므로 표준 발화(gTTS)와 공용 함수를 매번 복원합니다.

## 3-2. 노트북별 실험 가이드

| 노트북 | 실험 목적 | 핵심 절차 | 판단 기준 |
|---|---|---|---|
| **2-1** | ASR 원리 해부 | ①멜 스펙트로그램 관찰 ②모델 구조·토크나이저 ③greedy vs beam 극장 ④CER 직접 구현+jiwer 검증 ⑤initial_prompt·후처리 | CER 숫자보다 **원리**(입력→인코더→디코더→평가→개선) 이해 |
| **2-2** | 공정한 엔진 비교 | 시험지 10문항 → 벤치 하네스 → faster-whisper(fp16/int8) → turbo·small 등록 → 토너먼트 → asr_v4 어댑터 이식 | **예산 0.5s 통과자 중 noisy CER 최저** = 우승 |
| **2-3** | 프로덕션 모듈 | ASRConfig → preprocess → transcribe_safe → 스트리밍 → 환각 필터·후처리 → KoreanASR → SNR 5dB 스트레스 | **계약 검증 통과 + 예외 없이 완주** (숫자는 환경 의존) |
| **2-F** | DSP 밑바닥 | 합성 신호(220Hz+배음+chirp)로 프레이밍→DFT→필터뱅크→멜을 직접 구현 | 예측한 그림(가로줄+대각선) 일치 + **librosa np.allclose** |
| **2-R** | 스트리밍 | 청크 시뮬레이터 → VAD 엔드포인트 → LocalAgreement → 3대 지표 측정 → SNR 5dB | `final_latency_ms ≤ 500`, `rtf < 1.0`, 확정 뒤집힘 0회 |
| **2-D** | 다이어라이제이션 | 모의 2화자 통화(pitch -4) → VAD→ECAPA→클러스터링 → DER | DER 성분 해석: miss/FA=**VAD 문제**, confusion=**임베딩·군집 문제** |
| **2-V** | 리치 전사 | 태그 파싱 → CER/지연 → ITN → LID → 무음(`<|nospeech|>`) → BGM | NAR 속도(자릿수 차) + 감정 태그는 gTTS에 없음(평가 설계 교훈) |
| **2-W** | 파인튜닝·양자화 | 원본 turbo vs 한국어 파인튜닝(clean/SNR5) → float16 vs int8_float16 | 파인튜닝 효과는 **소음·실환경**에서 드러남; 양자화 손실은 측정해서 확인 |
| **2-S** | Qwen3-ASR | LID 자동감지 → 배치 전사 → SNR 5dB → 코드 스위칭 | 한국어·코드스위칭 강점 확인 (로그 확률 없음 유의) |
| **2-L** | full-duplex | perceived gap 계산 → 정책 FSM 시나리오 3종 → 위임 시뮬레이션 | EoU 대기 300/700/1200ms 변화, filler 커버 한계 |
| **2-C** | 한계 검증 | 기본 전사 → 무음·소음 환각 → LID 부재 → 코드 스위칭 | 모델 카드 Limitations를 **직접 재현** — VAD 게이트 필요성 입증 |

## 3-3. macOS에서 실습하는 법 (신규) 🍎

| 노트북 | Mac에서의 진행 방법 |
|---|---|
| 2-1 / 2-2 | faster-whisper로 동일 실습: `WhisperModel("base", device="cpu", compute_type="int8")`. openai-whisper는 MLX 대체 가능. CER·벤치 하네스는 그대로 |
| 2-3 | `CFG.device="cpu"`, `compute_type="int8"`로 교체 후 KoreanASR 재현. `free_gpu()` 대신 `del` (macOS는 통합 메모리) |
| 2-F / 2-D / 2-R / 2-L | **코드 수정 없음** (GPU 불필요·시뮬레이션). 2-D의 ECAPA만 speechbrain CPU로 느릴 수 있음 |
| 2-V | SenseVoice **GGUF(q8)** 로 1차 전사 재현 (원본은 funasr 패키지도 CPU 동작). 리치 전사 태그 그대로 확인 가능 |
| 2-W | 한국어 파인튜닝 turbo를 faster-whisper `int8`로 로드 — 양자화 실험의 정확도 축만 일부 차이 |
| 2-S | (비공식) 0.6B fp16은 통합메모리로 시도 가능하나 공식 macOS 지원 아님 — MLX 전용 whisper 우선 권장 |
| 2-L2 API 섹션 | `gpt-4o-transcribe`는 네트워크만 있으면 Mac에서 그대로 (키는 `getpass`로) |

**맥 전용 미니 실험 (권장)**: 같은 한국어 발화를 ①faster-whisper int8 ②mlx-whisper ③whisper.cpp로 각각 전사해 **지연·CER·메모리**를 비교하고, 2-2 토너먼트의 우승자를 Mac 조건에서 다시 판정해 보세요.

## 3-4. 실험 판단 기준 모음 (빠른 참조)

| 지표 | 목표값 | 출처 |
|---|---|---|
| CER | 도메인별 목표 설정 (합성음은 낮게, SNR 5dB는 올라감) | 2-1 |
| 지연(ASR 몫) | ≤ 500ms | 1-3 |
| RTF | < 1.0 (실시간 후보) | 2-2 |
| 재질문 신뢰도 | avg_logprob < -0.8 또는 no_speech_prob > 0.6 | 2-1/2-3 |
| first_partial_ms / final_latency_ms / rtf | 첫 부분 가설 빠를수록 / final ≤500ms / <1.0 | 2-R |
| DER | miss+FA=VAD·confusion=군집으로 진단 | 2-D |
| 퀴즈 상식 | 정규화 후 CER 예제: "배송조회부탁해요"(8자) vs "배송조희부탁해" → 2/8 = 25% | 2-1 |


# 4. 효율적 설계를 위한 아키텍처 🏛️

## 4-1. 데이터 계약이 모든 것을 지탱한다

엔진은 바뀌어도 **계약은 유지**. 1-4에서 정의한 계약을 2-2(어댑터)·2-3(모듈)·2-S~2-W(신엔진)가 전부 지킵니다.

```
[오픈엔진 6종 + API 엔진] ──어댑터──▶ {"utt_id","engine","text","language",
                                      "confidence_ok","avg_logprob","latency_ms"}
                                            │ 확장 필드 자유 (emotion/events...)
                                            ▼
                          LLM(5주차)·QA·VoC 분석 — 형식 몰라도 됨
```

## 4-2. 어댑터 + 팩토리 = 교체 지점이 한 곳

```
에이전트 ── create_engine(config) ──▶ {openai-whisper | faster-whisper | ...}
                 ▲ config['asr']['engine'] 한 줄
                 └ 기존 코드를 건드리지 않고 '심장'만 교체 (2-2 5장)
```

## 4-3. KoreanASR 파이프라인 (2-3 — 배치/파일 기준)

```
오디오 파일/배열
      │
      ▼ preprocess_audio        F1 어떤 샘플레이트·채널도 16kHz 모노로
      ▼ faster-whisper(base fp16, greedy)   NF2 어댑터 교체 가능
      ▼ judge_confidence        F3 미달 시 beam=5 재시도 1회 (2단 전략)
      ▼ is_hallucination        F4 말 비율·정형문구·반복으로 차단
      ▼ postprocess_ko          F6 교정 사전 + 숫자-단위 표기
      ▼ 계약 어댑터             F2 + NF1 지연 기록
      ▼
   {"text", "confidence_ok", ...}  ──▶ 5주차 LLM
```

## 4-4. 스트리밍 파이프라인 (2-R — 실시간/부분 결과)

```
청크 스트림(320ms) ──▶ 버퍼 누적 ──▶ [매 2청크] 재전사 ──▶ LocalAgreement ──▶ partial 이벤트
                        │                                        (두 가설 일치 접두어만 확정)
                        └─ VAD ─▶ 엔드포인트 ─▶ 최종 전사 ───────────────▶ final 이벤트(is_final=True)
```

- 확정 텍스트는 **절대 뒤집히지 않는다** → 6주차 LLM→TTS 즉시 소비 가능.
- `is_final=True` 이벤트는 배치 계약 키도 포함 → **하위 호환** (배치 후처리 코드 재사용).

## 4-5. 다이어라이제이션 파이프라인 (2-D)

```
오디오 → ①VAD → ②슬라이딩 윈도우 ECAPA 임베딩 → ③클러스터링(코사인)
       → ④다수결 라벨·스무딩 → 세그먼트 ──▶ 세그먼트별 ASR ──▶ "상담원:/고객:" 스크립트
                                    ▲
                        DER(miss/FA/confusion)로 채점
```

## 4-6. 캐스케이드 vs full-duplex (2-L) — 왜 본 과정은 캐스케이드인가

| 관점 | full-duplex(GPT-Live) | 캐스케이드(본 과정) |
|---|---|---|
| 체감 공백 | 수백 ms 미만 | 예산 관리 필요 (1.5s) |
| API 접근 | ❌ 아직 없음 | ✅ 전 구성요소 교체 가능 |
| 중간 텍스트 로그 | 제한적 | ✅ ASR 텍스트로 QA·VoC 분석 |
| 도메인 커스터마이즈 | 프롬프트 수준 | ✅ 교정 사전·후처리 |
| 규제·감사(콜센터 필수) | 어려움 | ✅ 단계별 관측 가능 |

> 캐스케이드에 이식 가능한 full-duplex 개념: **프레임 단위 판단·backchannel·선발화 filler**(LLM 응답 대기 중 TTS로 먼저 내보내기 = 5주차 'preamble 패턴').

## 4-7. macOS 배포 아키텍처 (신규) 🍎

```
[로컬 온디바이스]                     [하이브리드]                     [클라우드 API]
  SenseVoice GGUF q8                 faster-whisper large-v3-turbo   gpt-4o-transcribe
  (초경량, 리치 전사)                  int8 · mlx-whisper turbo        (2-L: 분당 $0.006)
      │                                   │                               │
  배터리·엣지 우선                     M시리즈 통합 메모리 활용        프라이버시·품질 최우선
  (1차 전사·보류음 감지)               (오프라인 실시간 통화)           (네트워크 필요)
```

| 선택 | 적합 시나리오 | 고려사항 |
|---|---|---|
| SenseVoice GGUF | 배터리/엣지, 상시 청취, 감정·이벤트 필요 | 정확도 한계, 한국어 지원 확인 |
| faster-whisper int8 | 오프라인 실시간 에이전트, 한국어 파인튜닝 활용 | 모델 1.5GB, 첫 로드 시간 |
| mlx-whisper | M시리즈 최고 속도, 간단 설치 | MLX 생태계(파인튜닝 체크포인트 일부 미지원) |
| API (gpt-4o-transcribe) | 품질·속도 즉시 확보 | 비용·네트워크·민감정보 |

## 4-8. 설계 원칙 요약 (이 5줄이 이번 2주차의 전부)

1. **계약은 최소 보장, 확장은 자유** — 필수 키만 검사, 새 능력은 확장 필드로 (2-V).
2. **정책과 코드 분리** — 임계값·모델명은 `ASRConfig`에 모으고 로직은 함수로 (2-3).
3. **죽지 않는 모듈** — 실패도 `status` 필드의 '데이터'로 보고 → 뒤 단계(LLM)가 우아한 복구 선택 (2-3).
4. **2단 전략** — 평소 greedy(빠른 길 먼저), 신뢰도 의심될 때만 beam(확실한 길은 필요할 때) (2-3).
5. **예산이 우선** — 예산을 넘는 정확도는 사용자에게 도달하지 못한다. 요구사항이 예산을 정하고, 벤치마크는 그 예산에서 가능한 품질을 보여준다 (2-2).
